# PSTAT 134 FINAL PROJECT
- Sid Revanuru
- Jayden Gould
- Jason Kim
- Clark Enge

## Movie Recomender System

### Pulling the data

In [ ]:
import os
import time
import pandas as pd
import requests

cache_file = "all_movie_data"

if os.path.exists(cache_file):
    movies_df = pd.read_pickle(cache_file)

else:
    API_TOKEN = # API token here

    headers = {
        "accept": "application/json",
        "Authorization": f'Bearer {API_TOKEN}'
    }
    movie_data =[]
    for page in range(1, 501):
        url = f"https://api.themoviedb.org/3/discover/movie?sort_by=popularity.desc&page={page}"
        response = requests.get(url, headers=headers)
        data = response.json()

        if "results" in data:
            for movie in data['results']:
                movie_id=movie['id']

                details_url = f"https://api.themoviedb.org/3/movie/{movie_id}?append_to_response=credits,keywords"
                details_response = requests.get(details_url, headers=headers)
                details_data = details_response.json()

                movie_dict = {
                    "id": movie_id,
                    'title': details_data['title'],
                    'overview': details_data['overview'],
                    'genres': [genre['name'] for genre in details_data.get('genres', [])],
                    'keywords': [kw['name'] for kw in details_data.get('keywords', {}).get('keywords', [])],
                    'cast': [actor['name'] for actor in details_data.get('credits', {}).get('cast', [])[:5]],
                    'vote_average': details_data['vote_average'],                  
                    'popularity': details_data['popularity']
                }

                movie_data.append(movie_dict)

                time.sleep(.10)

    movies_df = pd.DataFrame(movie_data)

    movies_df.to_pickle(cache_file)
movies_df.head()

,id,title,overview,genres,keywords,cast,vote_average,popularity
0,1226863,The Super Mario Galaxy Movie,Having thwarted Bowser's previous plot to marr...,"[Family, Comedy, Adventure, Fantasy, Animation]","[galaxy, friendship, sibling relationship, spa...","[Chris Pratt, Anya Taylor-Joy, Charlie Day, Ja...",6.732,443.9285
1,1007757,Swapped,"A small woodland creature and a majestic bird,...","[Adventure, Animation, Family, Fantasy]","[wolf, buddy, forest fire, woodlands, loving, ...","[Michael B. Jordan, Juno Temple, Tracy Morgan,...",7.892,409.3611
2,1318447,Apex,A grieving woman pushing her limits on a solo ...,"[Action, Thriller]","[canoe trip, rock climbing, nutcase, survival ...","[Charlize Theron, Taron Egerton, Eric Bana, Ca...",6.417,369.7562
3,10867,Malena,12-year-old Renato experiences three significa...,[Drama],"[prostitute, jealousy, sicily, italy, widow, w...","[Monica Bellucci, Giuseppe Sulfaro, Luciano Fe...",7.429,310.2717
4,1198994,Send Help,Two colleagues become stranded on a deserted i...,"[Horror, Thriller, Comedy]","[bullying, role reversal, survival, struggle f...","[Rachel McAdams, Dylan O'Brien, Edyll Ismail, ...",6.978,247.9714


We want this algorithm to look at large bodies of text instead different strings from different columns, causing the model to jump around. We will now take all of the text-based columns and create a new column combining all of them into one big block of text for each movie to make it easier to read for the computer.

In [35]:
import ast


# Helper function
def clean(list):
    for i in list:
       return [i.replace(" ", "") for i in list]

# Removing spaces form the values in each list column
for col in ['genres', 'cast', 'keywords']:
    movies_df[col] = movies_df[col].apply(clean)

# Turning the overview column into a list of words
movies_df['overview'] = movies_df['overview'].apply(lambda x: x.split() if isinstance(x, str) else [])

# Adding a new column that combines all the lists into one big list with all the information the computer needs
movies_df['tags'] = movies_df['overview'] + movies_df['genres'] + movies_df['cast'] + movies_df['keywords']

# New cleaned df with just the essential information
cleaned_movies_df = movies_df[['id', 'title', 'tags']].copy()

# Making the giant list a string
cleaned_movies_df['tags'] = cleaned_movies_df['tags'].apply(lambda x: " ".join([str(i) for i in x]) if isinstance(x, list) else "")
# Making all the values in the giant string lowercase
cleaned_movies_df['tags'] = cleaned_movies_df['tags'].apply(lambda x: x.lower())

cleaned_movies_df.head()

,id,title,tags
0,1226863,The Super Mario Galaxy Movie,having thwarted bowser's previous plot to marr...
1,1007757,Swapped,"a small woodland creature and a majestic bird,..."
2,1318447,Apex,a grieving woman pushing her limits on a solo ...
3,10867,Malena,12-year-old renato experiences three significa...
4,1198994,Send Help,two colleagues become stranded on a deserted i...
